[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/ex2_bakeoff.ipynb)

# ISBA 2411 - The Bake-Off: five classifiers, one leaderboard

**Week 3, Lecture 6.**

**Goal (~20 min, teams).** Your team trains ONE classifier on the shared review data, measures three numbers (held-out accuracy, training time, model size), and posts a row to the class leaderboard. Then we read the board together and find the frontier.

**The one rule.** Everyone uses the same train and test split, so the numbers compare. Do not touch the test set.

**How to run.** Run Section 0 once. Then run ONLY your team's assigned section. Then post your row to the class leaderboard.

### Team assignments (this class)

| Contender | Team(s) |
|---|---|
| A · Logistic regression | Team 1, Team 7 |
| B · FastText | Team 2 |
| C · spaCy textcat | Team 8 |
| D · AutoGluon | Team 5 |
| E · DistilBERT (needs GPU) | Team 4 |

Randomly assigned. Run **Section 0** first, then run **only your contender's** section, then paste your row into the leaderboard.

## Section 0. Shared setup (everyone runs this first)

In [ ]:
# Shared data. Loads a CSV bundled in the course repo, so there is no Hugging
# Face download to fail and no rate limits -- all teams get the identical split.
import pandas as pd, time

DATA_URL = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%203/rotten_tomatoes.csv"
try:
    df = pd.read_csv(DATA_URL)            # primary: read straight from GitHub
except Exception:
    df = pd.read_csv("rotten_tomatoes.csv")  # offline fallback: upload to Colab

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]
train_text = train["text"].tolist(); train_label = train["label"].tolist()
test_text  = test["text"].tolist();  test_label  = test["label"].tolist()
labels = {0: "negative", 1: "positive"}

print("SHARED DATA   train:", len(train_text), "  test:", len(test_text))
print("Everyone trains on this split and scores on this test set. Do not change it.")

def report(team, model, accuracy, train_seconds, size, ship):
    line = f"{team}\t{model}\t{accuracy:.4f}\t{train_seconds:.1f}\t{size}\t{ship}"
    print("\n" + "=" * 56)
    print("  LEADERBOARD ROW")
    print("=" * 56)
    print(f"  Team       : {team}")
    print(f"  Model      : {model}")
    print(f"  Accuracy   : {accuracy:.4f}")
    print(f"  Train time : {train_seconds:.1f} s")
    print(f"  Model size : {size}")
    print(f"  Ship it?   : {ship}")
    print("-" * 56)
    print("  Copy this tab-separated line into the shared sheet:")
    print("  " + line)

**Offline fallback:** Section 0 already reads the data straight from the course repo, so it normally just works. Only if the room has **no internet**, download `rotten_tomatoes.csv` from the repo's `Week 3` folder ahead of time, upload it to the Colab file panel, and run the cell below instead of Section 0. The `report()` helper above must have been defined first.

In [ ]:
# Offline only. Skip if Section 0 worked.
import pandas as pd
df = pd.read_csv("rotten_tomatoes.csv")   # uploaded to the Colab file panel
train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]
train_text = train["text"].tolist(); train_label = train["label"].tolist()
test_text  = test["text"].tolist();  test_label  = test["label"].tolist()
labels = {0: "negative", 1: "positive"}
print("train:", len(train_text), "  test:", len(test_text))

## Run ONLY your assigned section

Each section edits `TEAM` and `ship`, trains, and prints your leaderboard row.

### Contender A. Logistic regression (TF-IDF baseline)

In [ ]:
TEAM = "Team 1"   # shared: Team 1, Team 7 -- set this to YOUR team                         # <- your team
ship = "yes / no, because ..."           # <- fill in after you see the numbers

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(train_text); Xte = vec.transform(test_text)
clf = LogisticRegression(max_iter=1000).fit(Xtr, train_label)
secs = time.time() - t0

acc = accuracy_score(test_label, clf.predict(Xte))
size = f"{len(vec.vocabulary_):,} features (~{clf.coef_.nbytes/1e6:.2f} MB)"
report(TEAM, "LogReg + TF-IDF", acc, secs, size, ship)

### Contender B. FastText (a fast linear model over word and subword features)

Native FastText usually installs on Colab. If the binary misbehaves, this cell falls back to a FastText-style character n-gram linear model so your team still gets a comparable result.

In [ ]:
TEAM = "Team 2"
ship = "yes / no, because ..."

import time, os
from sklearn.metrics import accuracy_score

def _write_ft(path, texts, labs):
    with open(path, "w") as f:
        for t, l in zip(texts, labs):
            f.write(f"__label__{l} " + t.replace("\n", " ") + "\n")

try:
    try:
        import fasttext
    except ImportError:
        os.system("pip -q install fasttext-wheel")
        import fasttext
    _write_ft("ft_train.txt", train_text, train_label)
    t0 = time.time()
    m = fasttext.train_supervised("ft_train.txt", wordNgrams=2, epoch=25, lr=0.5, dim=50, bucket=200000, verbose=0)
    secs = time.time() - t0
    preds = [int(m.predict(t.replace("\n", " "))[0][0].replace("__label__", "")) for t in test_text]
    acc = accuracy_score(test_label, preds)
    m.save_model("ft_model.bin")
    size = f"{os.path.getsize('ft_model.bin')/1e6:.1f} MB (.bin)"
    model_name = "FastText"
except Exception as e:
    print("Native FastText was unavailable, using a FastText-style fallback:", type(e).__name__)
    from sklearn.feature_extraction.text import HashingVectorizer
    from sklearn.linear_model import LogisticRegression
    t0 = time.time()
    vec = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2**20, alternate_sign=False)
    clf = LogisticRegression(max_iter=1000).fit(vec.transform(train_text), train_label)
    secs = time.time() - t0
    acc = accuracy_score(test_label, clf.predict(vec.transform(test_text)))
    size = "char n-gram hashed (~8 MB)"
    model_name = "FastText-style (char n-gram)"

report(TEAM, model_name, acc, secs, size, ship)

### Contender C. spaCy textcat (spaCy's own classifier, inside a pipeline)

In [ ]:
TEAM = "Team 8"
ship = "yes / no, because ..."

import time, random
try:
    import spacy
except ImportError:
    import os; os.system("pip -q install spacy"); import spacy
from spacy.training import Example
from spacy.util import minibatch
from sklearn.metrics import accuracy_score

nlp = spacy.blank("en")
_bow = {"model": {"@architectures": "spacy.TextCatBOW.v3", "exclusive_classes": True,
                  "ngram_size": 2, "no_output_layer": False}}
try:
    textcat = nlp.add_pipe("textcat", config=_bow)
except Exception:
    _bow["model"]["@architectures"] = "spacy.TextCatBOW.v2"
    try:
        textcat = nlp.add_pipe("textcat", config=_bow)
    except Exception:
        textcat = nlp.add_pipe("textcat")
textcat.add_label("pos"); textcat.add_label("neg")

def _ex(t, l):
    return Example.from_dict(nlp.make_doc(t), {"cats": {"pos": l == 1, "neg": l == 0}})
train_ex = [_ex(t, l) for t, l in zip(train_text, train_label)]

t0 = time.time()
opt = nlp.initialize(lambda: train_ex)
for epoch in range(6):
    random.shuffle(train_ex)
    for batch in minibatch(train_ex, size=64):
        nlp.update(batch, sgd=opt)
secs = time.time() - t0

def _pred(t):
    c = nlp(t).cats
    return 1 if c["pos"] >= c["neg"] else 0
acc = accuracy_score(test_label, [_pred(t) for t in test_text])
size = f"{len(nlp.to_bytes())/1e6:.2f} MB (pipeline)"
report(TEAM, "spaCy textcat (bow)", acc, secs, size, ship)

### Contender D. AutoGluon (AutoML: give it a time budget, let it choose)

The install takes a few minutes. That cost is itself a real leaderboard data point.

In [ ]:
TEAM = "Team 5"
ship = "yes / no, because ..."

import os; os.system("pip -q install autogluon.tabular")
import time, pandas as pd
from autogluon.tabular import TabularPredictor

tr = pd.DataFrame({"text": train_text, "label": train_label})
te = pd.DataFrame({"text": test_text,  "label": test_label})

t0 = time.time()
predictor = TabularPredictor(label="label", eval_metric="accuracy").fit(tr, time_limit=120)
secs = time.time() - t0

ev = predictor.evaluate(te)
acc = ev["accuracy"] if isinstance(ev, dict) else float(ev)
try:
    size_mb = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, fs in os.walk(predictor.path) for f in fs) / 1e6
    size = f"{size_mb:.0f} MB (all models)"
except Exception:
    size = "ensemble on disk"
report(TEAM, "AutoGluon (120s budget)", acc, secs, size, ship)

### Contender E. DistilBERT (a small transformer, fine-tuned)

**Set a GPU runtime first:** Runtime, then Change runtime type, then GPU. This is the heaviest contender on purpose, and that cost is a real data point for the board.

In [ ]:
TEAM = "Team 4"
ship = "yes / no, because ..."

import os; os.system("pip -q install 'transformers>=4.40' datasets accelerate")
import time
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset
from sklearn.metrics import accuracy_score

name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
def _prep(texts, labs):
    d = Dataset.from_dict({"text": texts, "label": labs})
    return d.map(lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=64), batched=True)
d_tr = _prep(train_text, train_label); d_te = _prep(test_text, test_label)

model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
args = TrainingArguments(output_dir="db_out", num_train_epochs=1,
                         per_device_train_batch_size=16, per_device_eval_batch_size=32,
                         logging_steps=100, report_to="none")
def _metric(p):
    return {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))}
trainer = Trainer(model=model, args=args, train_dataset=d_tr, compute_metrics=_metric)

t0 = time.time(); trainer.train(); secs = time.time() - t0
acc = trainer.evaluate(d_te)["eval_accuracy"]
nparams = sum(p.numel() for p in model.parameters()) / 1e6
report(TEAM, "DistilBERT (1 epoch)", acc, secs, f"~{nparams:.0f}M params (~{nparams*4/1000:.0f} MB)", ship)

## Optional: Boost round — tune *without* touching the test set

Want a better number? You can tune your model — but there is one honest way to do it. Picking settings by repeatedly checking **test** accuracy and keeping whatever scores best leaks the test set through your choices (the train-on-test trap in disguise).

So: carve a **validation set out of the training data**, tune against *that*, and score the test set only **once**, at the very end, for your final row.

In [ ]:
# Carve a validation set out of the TRAINING data (test_* stays untouched).
from sklearn.model_selection import train_test_split

tr_sub_text, val_text, tr_sub_label, val_label = train_test_split(
    train_text, train_label, test_size=0.2, random_state=0, stratify=train_label)

print("tuning split ->  train:", len(tr_sub_text), "  validation:", len(val_text))
print("Tune using val_* only. Score test_* just once, for your final posted row.")

### How to use it

1. In your contender's cell, train on **`tr_sub_text` / `tr_sub_label`** and check accuracy on **`val_text` / `val_label`** while you experiment.
2. When happy, retrain on the **full `train_text`** and score **`test_text`** *once*.
3. Post your improved row — most boosts cost **time and/or size**, so re-post all three numbers.

**Knobs by contender:**

- **A · LogReg:** `ngram_range=(1, 3)`, tune `min_df` / `max_features`, `sublinear_tf=True`, tune `C=`, `class_weight="balanced"`
- **B · FastText:** `epoch=50`, tune `lr`, `wordNgrams=3`, bigger `dim=`, or `model.autotune`
- **C · spaCy:** more epochs, `spacy.TextCatEnsemble` architecture, or a `spacy-transformers` backbone
- **D · AutoGluon:** `time_limit=600`, `presets="best_quality"`
- **E · DistilBERT:** `num_train_epochs=2-3`, `max_length=128`, tune learning rate, or try `roberta-base`

A boost that raises accuracy but triples the cost has *moved along the frontier*, not beaten it.

## Post to the leaderboard, then we read it together

Paste your tab-separated row into the shared class leaderboard:

**[ISBA 2411 Bake-Off Leaderboard](https://docs.google.com/spreadsheets/d/18TS2fmeV0jqE_oJea1eUHdjuskm30kMLsoPfYsd7LcM/edit)**

Once the board is full, we plot accuracy against training time and find the frontier.

**Bring back an answer to:**
1. Which model would you actually ship for this task, and why?
2. Did the most accurate model justify its training time and size?
3. Where does your model sit on the accuracy-versus-cost frontier?

**Milestone 2.** The contender your team ran is a candidate for your project's one deliberate improvement. Reuse this code on your own data, and justify the choice on more than accuracy.

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*